## Carregamento dos dados tratados

Nesta etapa, carregamos o arquivo CSV contendo os dados que foram previamente tratados na Parte 1 do desafio.

O dataset já passou por um processo de limpeza e preparação, incluindo:

- remoção de inconsistências
- tratamento de valores ausentes
- padronização de categorias
- criação de novas variáveis relevantes

Com os dados já organizados, podemos prosseguir para as etapas de análise exploratória e modelagem.

In [ ]:
import pandas as pd

df = pd.read_csv("telecomx_dados_tratados.csv")

df.head()

## Remoção de colunas irrelevantes

Nesta etapa removemos colunas que não contribuem para a análise ou para a construção de modelos preditivos.

Colunas como **identificadores únicos**, por exemplo o `customerID`, não possuem valor preditivo, pois cada valor é exclusivo para um cliente e não representa um padrão que possa ajudar a prever a evasão.

Além disso, manter essas colunas pode prejudicar alguns algoritmos de machine learning, pois elas introduzem informações irrelevantes no modelo.

Portanto, essas colunas serão removidas do dataset antes das próximas etapas da análise.

In [ ]:
df = df.drop(columns=["customerID"])

In [ ]:
df.columns

In [ ]:
df.head()

## Codificação das variáveis categóricas

Algoritmos de machine learning trabalham apenas com valores numéricos. No entanto, o dataset contém diversas variáveis categóricas, como tipo de contrato, método de pagamento e serviço de internet.

Para tornar essas variáveis compatíveis com os modelos preditivos, utilizamos a técnica **One-Hot Encoding**.

O One-Hot Encoding transforma cada categoria em uma nova coluna binária (0 ou 1), indicando a presença ou ausência daquela categoria para cada observação.

Esse processo evita que o modelo interprete categorias como valores ordenados, o que poderia gerar interpretações incorretas.

In [ ]:
categorical_columns = df.select_dtypes(include=["object"]).columns
categorical_columns

In [ ]:
df = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

In [ ]:
## Proporção de clientes que evadiram

Nesta etapa analisamos a distribuição da variável alvo **Churn**, que indica se o cliente cancelou ou não o serviço.

O objetivo é calcular a proporção de clientes que **evadiram (Churn = 1)** em relação aos que **permaneceram ativos (Churn = 0)**.

Essa análise é importante porque muitos datasets possuem **desequilíbrio entre classes**. Quando uma classe é muito mais frequente que a outra, os modelos de machine learning podem ter dificuldade em aprender padrões da classe minoritária.

Portanto, identificar esse desequilíbrio ajuda a decidir se técnicas adicionais serão necessárias, como:

- balanceamento de classes
- oversampling
- undersampling

In [ ]:
categorical_columns = df.select_dtypes(include=["object"]).columns
categorical_columns

In [ ]:
df = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

## Correlação e Seleção de Variáveis

Nesta etapa analisamos a relação entre as variáveis numéricas do dataset e a variável alvo **Churn**.

A matriz de correlação permite identificar quais variáveis possuem maior associação com a evasão de clientes. Variáveis com maior correlação podem ser boas candidatas para modelos preditivos.

Também realizamos análises direcionadas entre variáveis importantes e a evasão, como:

- Tempo de contrato (tenure) × Evasão
- Total gasto (TotalCharges) × Evasão

Essas relações são visualizadas utilizando gráficos como **heatmap, boxplot e scatterplot**, permitindo observar padrões e tendências nos dados.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# matriz de correlação
plt.figure(figsize=(12,8))
corr = df.corr()

sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Matriz de Correlação")
plt.show()

# boxplot: tempo de contrato vs evasão
sns.boxplot(x="Churn", y="tenure", data=df)
plt.title("Tempo de contrato vs Evasão")
plt.show()

# boxplot: total gasto vs evasão
sns.boxplot(x="Churn", y="TotalCharges", data=df)
plt.title("Total gasto vs Evasão")
plt.show()

# scatterplot
sns.scatterplot(x="tenure", y="TotalCharges", hue="Churn", data=df)
plt.title("Tempo de contrato vs Total gasto")
plt.show()

## Modelagem Preditiva

Nesta etapa construímos modelos de machine learning para prever a evasão de clientes.

Primeiro dividimos os dados em conjuntos de **treino e teste**, permitindo avaliar o desempenho do modelo em dados que ele nunca viu.

Foram utilizados dois tipos de modelos:

- **Regressão Logística**: modelo linear amplamente utilizado para classificação binária. Esse modelo pode se beneficiar da normalização dos dados.
- **Random Forest**: modelo baseado em árvores de decisão que geralmente não exige normalização e consegue capturar relações não lineares.

A comparação entre modelos permite avaliar qual abordagem apresenta melhor desempenho para prever a evasão.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# separação X e y
X = df.drop("Churn", axis=1)
y = df["Churn"]

# divisão treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# normalização para regressão logística
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# modelo 1: regressão logística
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)

# modelo 2: random forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

## Avaliação dos Modelos

Para avaliar o desempenho dos modelos utilizamos métricas comuns em problemas de classificação:

- **Acurácia**: proporção de previsões corretas
- **Precisão**: proporção de previsões positivas que estão corretas
- **Recall**: capacidade do modelo identificar corretamente os casos positivos
- **F1-score**: equilíbrio entre precisão e recall
- **Matriz de confusão**: mostra detalhadamente os acertos e erros do modelo

Essas métricas permitem comparar os modelos e identificar qual apresenta melhor capacidade de prever a evasão de clientes.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# previsões
y_pred_log = log_model.predict(X_test_scaled)
y_pred_rf = rf_model.predict(X_test)

# avaliação regressão logística
print("Regressão Logística")
print(classification_report(y_test, y_pred_log))

print("Matriz de Confusão")
print(confusion_matrix(y_test, y_pred_log))

# avaliação random forest
print("\nRandom Forest")
print(classification_report(y_test, y_pred_rf))

print("Matriz de Confusão")
print(confusion_matrix(y_test, y_pred_rf))

## Importância das Variáveis

Após treinar os modelos, analisamos quais variáveis possuem maior influência na previsão da evasão.

No caso da **Regressão Logística**, os coeficientes indicam o impacto de cada variável na probabilidade de churn.

Já no **Random Forest**, a importância das variáveis é calculada com base em quanto cada variável contribui para reduzir a impureza nas árvores de decisão.

Essa análise ajuda a identificar os fatores mais relevantes que influenciam o cancelamento dos clientes.

In [ ]:
import pandas as pd

# coeficientes regressão logística
coef = pd.Series(log_model.coef_[0], index=X.columns)

coef.sort_values().plot(kind="barh", figsize=(8,6))
plt.title("Importância das Variáveis - Regressão Logística")
plt.show()

# importância random forest
importances = pd.Series(rf_model.feature_importances_, index=X.columns)

importances.sort_values().plot(kind="barh", figsize=(8,6))
plt.title("Importância das Variáveis - Random Forest")
plt.show()

## Conclusão

A análise permitiu identificar fatores importantes relacionados à evasão de clientes.

Variáveis como **tempo de contrato, valor total gasto e tipo de contrato** apresentaram forte relação com o churn.

Clientes com contratos mais curtos ou menor tempo de permanência tendem a apresentar maior probabilidade de cancelamento.

A comparação entre modelos mostrou diferenças de desempenho entre abordagens lineares e modelos baseados em árvores.

Com base nesses resultados, algumas estratégias de retenção podem ser propostas:

- incentivar contratos de longo prazo
- oferecer benefícios para clientes novos
- monitorar clientes com maior risco de cancelamento

Essas ações podem contribuir para reduzir a evasão e aumentar a retenção de clientes.